In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot"
)

DATA_DIR = PROJECT_DIR / "Data"

AI_RESULTS_PATH = (
    PROJECT_DIR
    / "Tests"
    / "ai_investigation_results.csv"
)

GROUND_TRUTH_PATH = (
    DATA_DIR
    / "ground_truth.csv"
)


# ============================================================
# LOAD DATA
# ============================================================

ai_results = pd.read_csv(
    AI_RESULTS_PATH
)

ground_truth = pd.read_csv(
    GROUND_TRUTH_PATH
)


# ============================================================
# MAP GROUND TRUTH
# ============================================================

ground_truth = ground_truth[
    ground_truth["true_scenario"] != "NORMAL"
].copy()


# ============================================================
# MERGE AI RESULTS WITH GROUND TRUTH
# ============================================================

evaluation = ai_results.merge(
    ground_truth[
        ["payment_id", "true_scenario"]
    ],
    on="payment_id",
    how="left"
)


# ============================================================
# CHECK FOR MISSING GROUND TRUTH
# ============================================================

missing_truth = evaluation[
    evaluation["true_scenario"].isna()
]

if len(missing_truth) > 0:

    print(
        "ERROR: Some AI results have no ground truth:"
    )

    print(
        missing_truth[
            ["payment_id"]
        ]
    )

    raise ValueError(
        "Missing ground truth records."
    )


# ============================================================
# CLASSIFICATION CORRECTNESS
# ============================================================

evaluation["correct"] = (
    evaluation["ai_classification"]
    == evaluation["true_scenario"]
)


# ============================================================
# OVERALL ACCURACY
# ============================================================

accuracy = (
    evaluation["correct"].mean()
)


print(
    f"\nAI classification accuracy: "
    f"{accuracy:.2%}"
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

print("\nConfusion matrix:")

confusion_matrix = pd.crosstab(
    evaluation["true_scenario"],
    evaluation["ai_classification"],
    rownames=["Actual"],
    colnames=["Predicted"],
    margins=True
)

print(
    confusion_matrix
)


# ============================================================
# PER-CLASS RESULTS
# ============================================================

print("\nPer-class accuracy:")

per_class = (
    evaluation
    .groupby("true_scenario")["correct"]
    .agg(
        total="count",
        correct="sum"
    )
)

per_class["accuracy"] = (
    per_class["correct"]
    / per_class["total"]
)

print(
    per_class
)


# ============================================================
# SHOW INCORRECT CASES
# ============================================================

incorrect = evaluation[
    ~evaluation["correct"]
].copy()


print(
    f"\nIncorrect classifications: "
    f"{len(incorrect)}"
)


if len(incorrect) > 0:

    print(
        incorrect[
            [
                "payment_id",
                "exception",
                "true_scenario",
                "ai_classification",
                "difference",
                "recommended_action",
                "confidence"
            ]
        ].to_string(index=False)
    )


# ============================================================
# SAVE EVALUATION
# ============================================================

evaluation_path = (
    PROJECT_DIR
    / "Tests"
    / "ai_evaluation.csv"
)

evaluation.to_csv(
    evaluation_path,
    index=False
)


print(
    f"\nEvaluation saved to:"
    f"\n{evaluation_path}"
)


AI classification accuracy: 98.78%

Confusion matrix:
Predicted             MISSING_BANK_RECORD  PARTIAL_REFUND  SETTLEMENT_DELAY  \
Actual                                                                        
MISSING_BANK_RECORD                    12               0                 1   
PARTIAL_REFUND                          0              17                 0   
SETTLEMENT_DELAY                        0               0                36   
UNEXPLAINED_MISMATCH                    0               0                 0   
All                                    12              17                37   

Predicted             UNEXPLAINED_MISMATCH  All  
Actual                                           
MISSING_BANK_RECORD                      0   13  
PARTIAL_REFUND                           0   17  
SETTLEMENT_DELAY                         0   36  
UNEXPLAINED_MISMATCH                    16   16  
All                                     16   82  

Per-class accuracy:
                    

In [7]:
import pandas as pd

results = pd.read_csv(
    r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Output\reconciliation_results.csv"
)

print(
    results[
        results["payment_id"] == "pay_0093"
    ].T
)

                                        92
payment_id                        pay_0093
gross_amount                         25000
payment_date                    2026-08-07
settlement_id                     set_0093
settlement_date                 2026-08-09
fee                                  500.0
tax                                   90.0
expected_net_amount                24410.0
settlement_net_amount              24410.0
bank_amount                            NaN
refund_amount                          NaN
delay_days                               2
status                           EXCEPTION
reason                 MISSING_BANK_RECORD


In [5]:
payments = pd.read_csv(
    r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\payments.csv"
)

settlements = pd.read_csv(
    r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\settlements.csv"
)

print("PAYMENT:")
print(
    payments[
        payments["payment_id"] == "pay_0093"
    ].T
)

print("\nSETTLEMENT:")
print(
    settlements[
        settlements["payment_id"] == "pay_0093"
    ].T
)

PAYMENT:
                      92
payment_id      pay_0093
order_id      order_0093
gross_amount       25000
payment_date  2026-08-07
status          captured

SETTLEMENT:
                         92
settlement_id      set_0093
payment_id         pay_0093
gross_amount          25000
fee                   500.0
tax                    90.0
net_amount          24410.0
settlement_date  2026-08-09


In [6]:
from pathlib import Path

ai_path = Path(
    r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Tests\ai_investigation_results.csv"
)

if ai_path.exists():
    ai_path.unlink()
    print("Old AI results deleted.")
else:
    print("No old AI results found.")

Old AI results deleted.


In [8]:
import pandas as pd

ground_truth = pd.read_csv(
    r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\ground_truth.csv"
)

print(ground_truth.head(10))
print()
print(ground_truth["true_scenario"].value_counts())

  payment_id         true_scenario
0   pay_0001      SETTLEMENT_DELAY
1   pay_0002      SETTLEMENT_DELAY
2   pay_0003                NORMAL
3   pay_0004   MISSING_BANK_RECORD
4   pay_0005      SETTLEMENT_DELAY
5   pay_0006        PARTIAL_REFUND
6   pay_0007        PARTIAL_REFUND
7   pay_0008        PARTIAL_REFUND
8   pay_0009      SETTLEMENT_DELAY
9   pay_0010  UNEXPLAINED_MISMATCH

true_scenario
SETTLEMENT_DELAY        36
NORMAL                  18
PARTIAL_REFUND          17
UNEXPLAINED_MISMATCH    16
MISSING_BANK_RECORD     13
Name: count, dtype: int64


In [10]:
import pandas as pd

AI_PATH = r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Tests\ai_investigation_results.csv"

ai = pd.read_csv(AI_PATH)

print(ai.columns.tolist())
print()
print(ai.head().to_string())

['payment_id', 'exception', 'ai_classification', 'explanation', 'difference', 'recommended_action', 'confidence']

  payment_id                      exception    ai_classification                                                                                                                                                                                                                                                     explanation  difference   recommended_action  confidence
0   pay_0001               SETTLEMENT_DELAY     SETTLEMENT_DELAY                                                                                             The payment of 1500 on 2026-08-01 settled with a delay of 10 days on 2026-08-11, resulting in a net settlement amount of 1464.60 after accounting for fees and tax.         0.0  WAIT_FOR_SETTLEMENT        1.00
1   pay_0002               SETTLEMENT_DELAY     SETTLEMENT_DELAY                  The payment of 5000 was settled after a delay of 7 days on 2026-08-09, 

In [11]:
import pandas as pd

AI_PATH = r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Tests\ai_investigation_results.csv"
GT_PATH = r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\ground_truth.csv"

ai = pd.read_csv(AI_PATH)
gt = pd.read_csv(GT_PATH)

evaluation = ai.merge(
    gt,
    on="payment_id",
    how="left"
)

print("Total AI investigations:", len(evaluation))
print()

print("AI classification counts:")
print(evaluation["ai_classification"].value_counts())
print()

print("Ground truth counts:")
print(evaluation["true_scenario"].value_counts())
print()

evaluation["correct"] = (
    evaluation["ai_classification"] == evaluation["true_scenario"]
)

correct = evaluation["correct"].sum()
incorrect = (~evaluation["correct"]).sum()
accuracy = evaluation["correct"].mean() * 100

print("Correct:", correct)
print("Incorrect:", incorrect)
print(f"Accuracy: {accuracy:.2f}%")
print()

print("INCORRECT CASES:")

incorrect_cases = evaluation.loc[
    ~evaluation["correct"],
    [
        "payment_id",
        "true_scenario",
        "exception",
        "ai_classification",
        "explanation",
        "difference",
        "recommended_action",
        "confidence",
    ]
]

print(incorrect_cases.to_string(index=False))

Total AI investigations: 82

AI classification counts:
ai_classification
SETTLEMENT_DELAY        38
PARTIAL_REFUND          17
UNEXPLAINED_MISMATCH    16
MISSING_BANK_RECORD     11
Name: count, dtype: int64

Ground truth counts:
true_scenario
SETTLEMENT_DELAY        36
PARTIAL_REFUND          17
UNEXPLAINED_MISMATCH    16
MISSING_BANK_RECORD     13
Name: count, dtype: int64

Correct: 80
Incorrect: 2
Accuracy: 97.56%

INCORRECT CASES:
payment_id       true_scenario           exception ai_classification                                                                                                                                                                                                                                                                                                                  explanation  difference  recommended_action  confidence
  pay_0086 MISSING_BANK_RECORD MISSING_BANK_RECORD  SETTLEMENT_DELAY                                                                

In [12]:
import pandas as pd

AI_PATH = r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Tests\ai_investigation_results.csv"
GT_PATH = r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\ground_truth.csv"

ai = pd.read_csv(AI_PATH)
gt = pd.read_csv(GT_PATH)

evaluation = ai.merge(
    gt,
    on="payment_id",
    how="left"
)

evaluation["correct"] = (
    evaluation["ai_classification"] == evaluation["true_scenario"]
)

correct = evaluation["correct"].sum()
total = len(evaluation)
accuracy = correct / total

print("=" * 50)
print("AI INVESTIGATION EVALUATION")
print("=" * 50)

print(f"Total investigations : {total}")
print(f"Correct classifications: {correct}")
print(f"Incorrect classifications: {total - correct}")
print(f"Accuracy              : {accuracy:.2%}")

print("\nClassification results:")
print(
    pd.crosstab(
        evaluation["true_scenario"],
        evaluation["ai_classification"]
    )
)

print("\nIncorrect cases:")
print(
    evaluation.loc[
        ~evaluation["correct"],
        [
            "payment_id",
            "true_scenario",
            "ai_classification",
            "difference",
            "recommended_action",
            "confidence"
        ]
)

AI INVESTIGATION EVALUATION
Total investigations : 82
Correct classifications: 80
Incorrect classifications: 2
Accuracy              : 97.56%

Classification results:
ai_classification     MISSING_BANK_RECORD  PARTIAL_REFUND  SETTLEMENT_DELAY  \
true_scenario                                                                 
MISSING_BANK_RECORD                    11               0                 2   
PARTIAL_REFUND                          0              17                 0   
SETTLEMENT_DELAY                        0               0                36   
UNEXPLAINED_MISMATCH                    0               0                 0   

ai_classification     UNEXPLAINED_MISMATCH  
true_scenario                               
MISSING_BANK_RECORD                      0  
PARTIAL_REFUND                           0  
SETTLEMENT_DELAY                         0  
UNEXPLAINED_MISMATCH                    16  

Incorrect cases:
payment_id       true_scenario ai_classification  difference  recommen